## Phase 4, Lesson 1 — Testing Assumptions

In [1]:
import numpy as np
from scipy import stats

np.random.seed(42)

normal_data = np.random.normal(loc=50, scale=10, size=40)
skewed_data = np.random.exponential(scale=10, size=40)

# --- Shapiro-Wilk ---
stat_n, p_n = stats.shapiro(normal_data)
stat_s, p_s = stats.shapiro(skewed_data)

print("SHAPIRO-WILK TEST")
print(f"  Normal data:  W={stat_n:.4f}, p={p_n:.4f} → {'non-normal' if p_n<=0.05 else 'normal'}")
print(f"  Skewed data:  W={stat_s:.4f}, p={p_s:.4f} → {'non-normal' if p_s<=0.05 else 'normal'}")

print()

# --- Levene test for equal variances ---
group1 = np.random.normal(loc=50, scale=5,  size=30)
group2 = np.random.normal(loc=55, scale=15, size=30)

lev_stat, p_lev = stats.levene(group1, group2)
print("LEVENE TEST (equal variances)")
print(f"  group1 std: {group1.std():.2f}")
print(f"  group2 std: {group2.std():.2f}")
print(f"  p={p_lev:.4f} → {'unequal variances' if p_lev<=0.05 else 'equal variances'}")
print()

# --- What to do when variances are unequal ---
# Pass equal_var=False to ttest_ind (Welch's t-test)
t, p = stats.ttest_ind(group1, group2, equal_var=False)
print(f"  Welch t-test (equal_var=False): p={p:.4f}")

SHAPIRO-WILK TEST
  Normal data:  W=0.9792, p=0.6604 → normal
  Skewed data:  W=0.8623, p=0.0002 → non-normal

LEVENE TEST (equal variances)
  group1 std: 4.66
  group2 std: 15.42
  p=0.0002 → unequal variances

  Welch t-test (equal_var=False): p=0.0511


## Phase 4, Lesson 2 — Effect Size and Confidence Intervals

Cohen's d measures how many standard deviations apart two means are:  
d = (mean1 - mean2) / pooled standard deviation  
A narrow CI means your estimate is precise. A wide CI means high uncertainty — you need more data.

In [2]:
import numpy as np
from scipy import stats

np.random.seed(42)

control   = np.random.normal(loc=50, scale=10, size=40)
treatment = np.random.normal(loc=56, scale=10, size=40)

# --- t-test ---
t_stat, p_val = stats.ttest_ind(control, treatment)
print(f"t-test: p = {p_val:.4f}")
print()

# --- Cohen's d ---
pooled_std = np.sqrt((control.std()**2 + treatment.std()**2) / 2)
cohens_d   = (treatment.mean() - control.mean()) / pooled_std

print(f"EFFECT SIZE")
print(f"  Control mean:   {control.mean():.2f}")
print(f"  Treatment mean: {treatment.mean():.2f}")
print(f"  Cohen's d:      {cohens_d:.4f}")

if abs(cohens_d) < 0.2:
    label = "negligible"
elif abs(cohens_d) < 0.5:
    label = "small"
elif abs(cohens_d) < 0.8:
    label = "medium"
else:
    label = "large"
print(f"  Interpretation: {label} effect")
print()

# --- 95% Confidence interval for the difference in means ---
diff        = treatment.mean() - control.mean()
se_diff     = np.sqrt(control.std()**2/len(control) + treatment.std()**2/len(treatment))
df          = len(control) + len(treatment) - 2
t_critical  = stats.t.ppf(0.975, df=df)   # two-tailed 95%

ci_lower = diff - t_critical * se_diff
ci_upper = diff + t_critical * se_diff

print(f"CONFIDENCE INTERVAL (95%)")
print(f"  Mean difference: {diff:.2f}")
print(f"  95% CI: ({ci_lower:.2f}, {ci_upper:.2f})")
print()
print(f"  Interpretation: the true difference in means is plausibly")
print(f"  between {ci_lower:.2f} and {ci_upper:.2f} points.")

t-test: p = 0.0004

EFFECT SIZE
  Control mean:   47.81
  Treatment mean: 55.71
  Cohen's d:      0.8340
  Interpretation: large effect

CONFIDENCE INTERVAL (95%)
  Mean difference: 7.90
  95% CI: (3.68, 12.11)

  Interpretation: the true difference in means is plausibly
  between 3.68 and 12.11 points.


## Phase 4, Lesson 3 — Building a Reusable Analysis Function

In [3]:
import numpy as np
from scipy import stats

def compare_two_groups(group1, group2, group1_name="Group 1", group2_name="Group 2", alpha=0.05):
    """
    Full two-group comparison workflow.
    Automatically tests assumptions and selects the appropriate test.
    Returns a dictionary of results.
    """

    results = {}

    # ------------------------------------------------
    # Step 1: Descriptive statistics
    # ------------------------------------------------
    results['group1_mean'] = group1.mean()
    results['group2_mean'] = group2.mean()
    results['group1_std']  = group1.std()
    results['group2_std']  = group2.std()
    results['group1_n']    = len(group1)
    results['group2_n']    = len(group2)

    # ------------------------------------------------
    # Step 2: Test normality (Shapiro-Wilk)
    # ------------------------------------------------
    _, p_norm1 = stats.shapiro(group1)
    _, p_norm2 = stats.shapiro(group2)
    results['normal_group1'] = p_norm1 > alpha
    results['normal_group2'] = p_norm2 > alpha
    both_normal = results['normal_group1'] and results['normal_group2']

    # ------------------------------------------------
    # Step 3: Test equal variances (Levene)
    # ------------------------------------------------
    _, p_lev = stats.levene(group1, group2)
    results['equal_variance'] = p_lev > alpha

    # ------------------------------------------------
    # Step 4: Choose and run the right test
    # ------------------------------------------------
    if not both_normal:
        results['test_used'] = 'Mann-Whitney U'
        stat, p = stats.mannwhitneyu(group1, group2, alternative='two-sided')
    elif not results['equal_variance']:
        results['test_used'] = 'Welch t-test'
        stat, p = stats.ttest_ind(group1, group2, equal_var=False)
    else:
        results['test_used'] = 'Independent t-test'
        stat, p = stats.ttest_ind(group1, group2, equal_var=True)

    results['statistic'] = stat
    results['p_value']   = p
    results['reject_h0'] = p <= alpha

    # ------------------------------------------------
    # Step 5: Effect size (Cohen's d)
    # ------------------------------------------------
    pooled_std       = np.sqrt((group1.std()**2 + group2.std()**2) / 2)
    results['cohens_d'] = (group2.mean() - group1.mean()) / pooled_std

    # ------------------------------------------------
    # Step 6: 95% Confidence interval for difference
    # ------------------------------------------------
    diff   = group2.mean() - group1.mean()
    se     = np.sqrt(group1.std()**2/len(group1) + group2.std()**2/len(group2))
    df     = len(group1) + len(group2) - 2
    t_crit = stats.t.ppf(1 - alpha/2, df=df)
    results['ci_lower'] = diff - t_crit * se
    results['ci_upper'] = diff + t_crit * se

    # ------------------------------------------------
    # Step 7: Print clean summary
    # ------------------------------------------------
    print("=" * 55)
    print(f"  COMPARISON: {group1_name} vs {group2_name}")
    print("=" * 55)
    print(f"  {group1_name}: mean={results['group1_mean']:.2f}, std={results['group1_std']:.2f}, n={results['group1_n']}")
    print(f"  {group2_name}: mean={results['group2_mean']:.2f}, std={results['group2_std']:.2f}, n={results['group2_n']}")
    print()
    print(f"  Normality:  {group1_name}={'normal' if results['normal_group1'] else 'non-normal'}, "
          f"{group2_name}={'normal' if results['normal_group2'] else 'non-normal'}")
    print(f"  Variances:  {'equal' if results['equal_variance'] else 'unequal'}")
    print(f"  Test used:  {results['test_used']}")
    print()
    print(f"  Statistic:  {results['statistic']:.4f}")
    print(f"  p-value:    {results['p_value']:.4f}")
    print(f"  Decision:   {'REJECT H0' if results['reject_h0'] else 'FAIL TO REJECT H0'}")
    print()
    print(f"  Cohen's d:  {results['cohens_d']:.4f}")
    print(f"  95% CI for difference: ({results['ci_lower']:.2f}, {results['ci_upper']:.2f})")
    print("=" * 55)

    return results

# ------------------------------------------------
# Test it on three scenarios
# ------------------------------------------------
np.random.seed(42)

print("\n--- Scenario 1: Normal data, equal variances ---\n")
a1 = np.random.normal(50, 10, 40)
b1 = np.random.normal(56, 10, 40)
r1 = compare_two_groups(a1, b1, "Control", "Treatment")

print("\n--- Scenario 2: Normal data, unequal variances ---\n")
a2 = np.random.normal(50, 5,  40)
b2 = np.random.normal(56, 20, 40)
r2 = compare_two_groups(a2, b2, "Low variance", "High variance")

print("\n--- Scenario 3: Non-normal data ---\n")
a3 = np.random.exponential(10, 40)
b3 = np.random.exponential(15, 40)
r3 = compare_two_groups(a3, b3, "Exp group A", "Exp group B")


--- Scenario 1: Normal data, equal variances ---

  COMPARISON: Control vs Treatment
  Control: mean=47.81, std=9.41, n=40
  Treatment: mean=55.71, std=9.53, n=40

  Normality:  Control=normal, Treatment=normal
  Variances:  equal
  Test used:  Independent t-test

  Statistic:  -3.6829
  p-value:    0.0004
  Decision:   REJECT H0

  Cohen's d:  0.8340
  95% CI for difference: (3.68, 12.11)

--- Scenario 2: Normal data, unequal variances ---

  COMPARISON: Low variance vs High variance
  Low variance: mean=50.05, std=4.26, n=40
  High variance: mean=55.34, std=19.36, n=40

  Normality:  Low variance=normal, High variance=normal
  Variances:  unequal
  Test used:  Welch t-test

  Statistic:  -1.6666
  p-value:    0.1029
  Decision:   FAIL TO REJECT H0

  Cohen's d:  0.3774
  95% CI for difference: (-0.95, 11.53)

--- Scenario 3: Non-normal data ---

  COMPARISON: Exp group A vs Exp group B
  Exp group A: mean=11.83, std=10.29, n=40
  Exp group B: mean=15.98, std=13.53, n=40

  Normality